# RS-Code-SSM: EpistemicUnit Generation on Kaggle T4

Generates 15,000+ EpistemicUnits (EUs) for the EpiChat knowledge graph using
Ollama + llama3.1:8b on GPU. Uploads results to HuggingFace Dataset when done.

## Setup
1. Enable GPU: **Settings → Accelerator → GPU T4 x2**
2. Enable Internet: **Settings → Internet → On**
3. Add secret: `HF_TOKEN` (HuggingFace write token)
4. Run all cells

**Expected runtime**: ~4–6 hours for 15,000 EUs

In [1]:
# ── Cell 1: Install Ollama + dependencies ─────────────────────────────────────
import subprocess, sys, os

# Install Ollama
subprocess.run('sudo apt-get install zstd', shell=True, check=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

# Install Python deps (epichat uses sentence-transformers for embeddings)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface_hub', 'faiss-cpu', 'numpy', 'scipy', 'sentence-transformers'
], check=True)

print('Done.')

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 134 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]


debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 603 kB in 1s (671 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 124463 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.3 MB/s eta 0:00:00
Done.


In [2]:
# ── Cell 2: Start Ollama + pull model ─────────────────────────────────────────
import subprocess, time

# Start Ollama server in background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Pull llama3.1:8b
subprocess.run(['ollama', 'pull', 'llama3.1:8b'], check=True)
print('Model ready.')

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠋ pulling manifest ⠋ pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  47 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   3% ▕                  ▏ 136 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   4% ▕                  ▏ 180 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   6% ▕█                 ▏ 278 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   8% ▕█                 ▏ 376 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   9% ▕█                 ▏ 422 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  10% ▕█                 ▏ 514 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  12% ▕██                ▏ 604 MB/4.9 GB                  pulling manifes

Model ready.


pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest ⠹ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest ⠸ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2

In [3]:
# ── Cell 3: Clone repo (epichat is in-repo, no separate EpiChat clone) ─────────
from pathlib import Path

if not Path('RS-Code-SSM').exists():
    subprocess.run(['git', 'clone', 'https://github.com/pgalyen1987/RS-Code-SSM.git'], check=True)

os.chdir('RS-Code-SSM')
sys.path.insert(0, '.')

print('Repo ready. epichat/ is in-repo.')

Cloning into 'RS-Code-SSM'...


Fetching origin
HEAD is now at 1a44264 trying to fix import error on kaggle still


Cloning into '../EpiChat'...


Repos ready.


In [4]:
# ── Cell 4: Config (epichat is in-repo at REPO_ROOT/epichat) ───────────────────
import os
from pathlib import Path

HF_TOKEN  = os.environ.get('HF_TOKEN', '')       # set in Kaggle Secrets
DATA_REPO = 'pgalyen1987/rs-code-ssm-traces'     # HF dataset repo
REPO_ROOT = os.getcwd()
EPICHAT_DIR = os.path.join(REPO_ROOT, 'epichat')
OUTPUT_DIR  = os.path.join(EPICHAT_DIR, 'episteme_data')
MODEL       = 'llama3.1:8b'
TARGET_EUS  = 15000

# Set env vars for scripts
os.environ['EPICHAT_DIR'] = EPICHAT_DIR
os.environ['REPO_ROOT'] = REPO_ROOT
os.environ['TMP_DIR'] = os.environ.get('TMP_DIR', '/tmp')

# Ensure episteme_data exists (replace broken symlink if present)
episteme = Path(OUTPUT_DIR)
if episteme.is_symlink() and not episteme.exists():
    episteme.unlink()
episteme.mkdir(parents=True, exist_ok=True)
print(f'Config ready. Target: {TARGET_EUS} EUs')
print(f'EPICHAT_DIR={EPICHAT_DIR}')

Config ready. Target: 15000 EUs
EPICHAT_DIR=/kaggle/working/EpiChat


In [5]:
# ── Cell 5: Run EU generation (epichat in-repo, env vars set in Cell 4) ───────
result = subprocess.run(
    [sys.executable, 'scripts/generate_eus.py'],
    cwd=REPO_ROOT,
    env={**os.environ, 'EPICHAT_DIR': EPICHAT_DIR, 'REPO_ROOT': REPO_ROOT},
    check=False
)
print(f'EU generation exit code: {result.returncode}')

EU generation exit code: 1


Traceback (most recent call last):
  File "/kaggle/working/RS-Code-SSM/scripts/generate_eus.py", line 33, in <module>
    from epichat.core.knowledge_graph import KnowledgeGraph
ModuleNotFoundError: No module named 'epichat'


In [6]:
# ── Cell 6: Check results ─────────────────────────────────────────────────────
import json
from pathlib import Path

units_file = Path(OUTPUT_DIR) / 'units.json'
if units_file.exists():
    with open(units_file) as f:
        units = json.load(f)
    print(f'Total EUs: {len(units)}')
    for uid, u in list(units.items())[:3]:
        print(f'  [{uid[:8]}] {str(u.get("proposition",""))[:80]}')
else:
    print('units.json not found')

Total EUs: 10876
  [ddfe86d0] A statement cannot be both true and false simultaneously
  [99fb542b] If A implies B, and A is true, then B is true (modus ponens)
  [26dd9cb4] Everything is identical to itself (law of identity)


In [7]:
# ── Cell 7: Export training traces from EUs ───────────────────────────────────
result = subprocess.run([
    sys.executable, '-m', 'train.epichat_export',
    '--epichat-dir', EPICHAT_DIR,
    '--output', 'data/epichat_traces.jsonl',
], check=False)
print(f'Export exit code: {result.returncode}')

if Path('data/epichat_traces.jsonl').exists():
    n = sum(1 for _ in open('data/epichat_traces.jsonl'))
    print(f'Exported {n} SFT traces')

[INFO] Loaded 10876 EpistemicUnits from EpiChat


Export exit code: 0
Exported 10669 SFT traces


[DONE] 10669 training traces written to data/epichat_traces.jsonl
       from 10570 unique EpistemicUnits
[INFO] Total: 10669 traces


In [8]:
# ── Cell 8: Upload to HuggingFace ─────────────────────────────────────────────
from huggingface_hub import HfApi
from pathlib import Path

if HF_TOKEN and DATA_REPO:
    api = HfApi(token=HF_TOKEN)
    try:
        api.create_repo(DATA_REPO, repo_type='dataset', exist_ok=True)
    except Exception as e:
        print(f'Repo: {e}')

    api.upload_file(
        path_or_fileobj=str(Path(OUTPUT_DIR) / 'units.json'),
        path_in_repo='episteme_data/units.json',
        repo_id=DATA_REPO, repo_type='dataset', token=HF_TOKEN,
    )
    print('Uploaded units.json')

    if Path('data/epichat_traces.jsonl').exists():
        api.upload_file(
            path_or_fileobj='data/epichat_traces.jsonl',
            path_in_repo='epichat_traces.jsonl',
            repo_id=DATA_REPO, repo_type='dataset', token=HF_TOKEN,
        )
        print('Uploaded epichat_traces.jsonl')

    print(f'Done: https://huggingface.co/datasets/{DATA_REPO}')
else:
    print('Set HF_TOKEN in Kaggle Secrets to upload.')

Set HF_TOKEN in Kaggle Secrets to upload.
